In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
import random

%matplotlib inline

In [ ]:
words = open("names.txt", "r").read().splitlines()
len(words)

32033

In [4]:
# shuffle up the words
random.seed(42)
random.shuffle(words)

In [ ]:
# get distinct chars across all words -- build vocab
chars = sorted(list(set("".join(words))))

# map chars to and from integers
# 0th char will be "." which denotes start/end of a word
char_to_int = {ch: idx + 1 for idx, ch in enumerate(chars)}
char_to_int["."] = 0
int_to_char = {idx: ch for ch, idx in char_to_int.items()}

print(char_to_int)
print(int_to_char)

{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'o': 15, 'p': 16, 'q': 17, 'r': 18, 's': 19, 't': 20, 'u': 21, 'v': 22, 'w': 23, 'x': 24, 'y': 25, 'z': 26, '.': 0}
{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
# build train, eval and test datasets
# X: 3 chars --> Y: next char
# For word "emma"
# ... --> e
# ..e --> m
# .em --> m
# emm --> a
# mma --> .

block_size = 8  # number of chars we take to predict the next one


def get_dataset(words):
    X = []
    y = []
    for w in words:
        block = [0] * block_size  # we start a block with all "."
        # iterate over every character in the word plus "." to mark the end
        # plus "." is necessary or else the loop will completely skip last char of the word
        for ch in w + ".":
            ix = char_to_int[ch]
            y.append(ix)
            X.append(block)
            block = block[1:] + [ix]  # move the window forward
    X = torch.tensor(X)
    y = torch.tensor(y)
    return X, y


X, y = get_dataset(words[:10])


In [30]:
for xout, yout in zip(X[:25], y[:25]):
    xout_str = "".join([int_to_char[ch.item()] for ch in xout])
    yout_str = int_to_char[yout.item()]
    print("{} --> {}".format(xout_str, yout_str))

........ --> y
.......y --> u
......yu --> h
.....yuh --> e
....yuhe --> n
...yuhen --> g
..yuheng --> .
........ --> d
.......d --> i
......di --> o
.....dio --> n
....dion --> d
...diond --> r
..diondr --> e
.diondre --> .
........ --> x
.......x --> a
......xa --> v
.....xav --> i
....xavi --> e
...xavie --> n
..xavien --> .
........ --> j
.......j --> o
......jo --> r


In [16]:
y.item

<function Tensor.item()>